# 03. MAS-Rec 모델 평가

논문 학습 과정에서 저장한 MAS-Rec 가중치를 불러와 Test 데이터 전체를 예측합니다.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Library

In [3]:
# 데이터 처리
import os
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# BERT
from transformers import AutoModel, AutoTokenizer

# 전처리 및 평가
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


device: cuda
gpu: Tesla T4


## Utils


In [6]:
class ReviewPairDataset(Dataset):
    def __init__(self, user_texts, item_texts, user_ids, item_ids, ratings, tokenizer, max_length=512):
        self.user_texts = user_texts
        self.item_texts = item_texts
        self.user_ids = user_ids
        self.item_ids = item_ids
        self.ratings = ratings
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.user_texts)

    def __getitem__(self, idx):
        user_enc = self.tokenizer(self.user_texts[idx], truncation=True, padding="max_length", max_length=self.max_length, return_tensors="pt")
        item_enc = self.tokenizer(self.item_texts[idx], truncation=True, padding="max_length", max_length=self.max_length, return_tensors="pt")
        user_id = torch.tensor(self.user_ids[idx], dtype=torch.long)
        item_id = torch.tensor(self.item_ids[idx], dtype=torch.long)
        rating = torch.tensor(float(self.ratings[idx]), dtype=torch.float)

        return {
            "input_ids_user": user_enc["input_ids"].squeeze(0),
            "attention_mask_user": user_enc["attention_mask"].squeeze(0),
            "input_ids_item": item_enc["input_ids"].squeeze(0),
            "attention_mask_item": item_enc["attention_mask"].squeeze(0),
            "labels": rating,
            "user_id": user_id,
            "item_id": item_id
        }


In [7]:
class Co_Attention_Layer(nn.Module):
    def __init__(self, emb_dim=768, nhead=8):
        super().__init__()
        self.attn_user_to_item = nn.MultiheadAttention(embed_dim=emb_dim, num_heads=nhead, batch_first=True)
        self.attn_item_to_user = nn.MultiheadAttention(embed_dim=emb_dim, num_heads=nhead, batch_first=True)

    def forward(self, user_vec, item_vec):
        # 입력: [batch, emb]
        user_vec = user_vec.unsqueeze(1)  # [batch, 1, emb]
        item_vec = item_vec.unsqueeze(1)  # [batch, 1, emb]

        # ① user attends to item (Q=user, K/V=item)
        attn_u2i, _ = self.attn_user_to_item(query=user_vec, key=item_vec, value=item_vec)  # [batch, 1, emb]

        # ② item attends to user (Q=item, K/V=user)
        attn_i2u, _ = self.attn_item_to_user(query=item_vec, key=user_vec, value=user_vec)  # [batch, 1, emb]

        # Flatten and concat both directions
        concat = torch.cat([attn_u2i.squeeze(1), attn_i2u.squeeze(1)], dim=-1)  # [batch, emb*2]
        return concat

## Load Dataset

In [8]:
# 데이터셋 불러오기
data_path = '/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v6_final.pkl'
df = pd.read_pickle(data_path)

print(df.shape)

(66718, 12)


In [9]:
# 원본 ID는 오류 분석 파일에만 보관
df['user_id_raw'] = df['user_id']
df['asin_raw'] = df['asin']

# 사용자와 상품 ID Encoding
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

df['user_id'] = user_encoder.fit_transform(df['user_id'])
df['asin'] = item_encoder.fit_transform(df['asin'])

n_users = df['user_id'].nunique()
n_items = df['asin'].nunique()

Num_User = n_users + 1
Num_Item = n_items + 1

print(n_users, n_items)

7449 16099


In [10]:
# train, valid, test으로 데이터셋 분리
train, test = train_test_split(df, test_size=0.2, random_state=42)
train, valid = train_test_split(train, test_size=0.125, random_state=42)

# 원본 행 번호 보관
test_source_index = test.index.to_numpy()

train = train.reset_index(drop=True)
valid = valid.reset_index(drop=True)
test = test.reset_index(drop=True)

print(train.shape, valid.shape, test.shape)

(46702, 14) (6672, 14) (13344, 14)


## Model

In [11]:
def make_mlp(sizes, activation=nn.ReLU):
    layers = []
    for i in range(len(sizes) - 1):
        layers.append(nn.Linear(sizes[i], sizes[i + 1]))
        layers.append(activation())
    return nn.Sequential(*layers)

In [12]:
def make_last_mlp(sizes, activation=nn.ReLU):
    layers = []
    for i in range(len(sizes) - 1):
        layers.append(nn.Linear(sizes[i], sizes[i + 1]))
        if i < len(sizes) - 2:
            layers.append(activation())
    return nn.Sequential(*layers)

In [13]:
class MAS_Rec(nn.Module):
    def __init__(self, n_users, n_items, user_model, item_model, nhead=8):
        super().__init__()

        # Interaction IDs
        self.user_id = nn.Embedding(n_users, 64)
        self.item_id = nn.Embedding(n_items, 64)

        # Two Tower Bert
        self.user_model = user_model
        self.item_model = item_model

        # [CLS] Dense
        self.cls_dense = make_mlp([768, 512, 256])

        # Co-Attention (bi-directional cross attention)
        # 256([CLS] dense output emb) + 64(interaction emb)
        self.cross_attn = Co_Attention_Layer(emb_dim=320, nhead=nhead)

        # Last Dense
        self.fc_final = make_last_mlp([640, 256, 1])

    def forward(self, user_id, item_id, input_ids_user, attention_mask_user, input_ids_item, attention_mask_item, labels=None):

        # Interaction IDs
        user_emb = self.user_id(user_id)    # [batch, 64]
        item_emb = self.item_id(item_id)    # [batch, 64]

        # Two Tower BERT
        # 여기에서 input_ids는 token_id(token index)
        user_cls = self.user_model(input_ids=input_ids_user, attention_mask=attention_mask_user).last_hidden_state[:, 0, :]
        item_cls = self.item_model(input_ids=input_ids_item, attention_mask=attention_mask_item).last_hidden_state[:, 0, :]

        # [CLS] Dense
        user_cls = self.cls_dense(user_cls)
        item_cls = self.cls_dense(item_cls)

        # Concat
        user_feat = torch.cat([user_emb, user_cls], dim=-1)
        item_feat = torch.cat([item_emb, item_cls], dim=-1)

        # Co-Attetnion
        co_attn_output = self.cross_attn(user_feat, item_feat)  # [batch, 512]

        # Last Dense Layer
        logits = self.fc_final(co_attn_output).squeeze(1)

        if labels is not None:
            loss = nn.MSELoss()(logits, labels)
            return {"loss": loss, "logits": logits}
        return {"logits": logits}

## 가중치 복원

사용자·상품 임베딩을 포함한 전체 `state_dict`를 불러옵니다.


In [14]:
# 저장된 가중치와 토크나이저 불러오기
model_path = '/content/drive/MyDrive/MAS_Rec/MAS_Rec.pth'
tokenizer_path = '/content/drive/MyDrive/MAS_Rec/MAS_Rec_Tokenizer'

tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

BERT_review = AutoModel.from_pretrained('bert-base-uncased')
BERT_image = AutoModel.from_pretrained('bert-base-uncased')

model = MAS_Rec(
    Num_User,
    Num_Item,
    user_model=BERT_review,
    item_model=BERT_image,
    nhead=8
)

try:
    state_dict = torch.load(model_path, map_location='cpu', weights_only=True)
except TypeError:
    state_dict = torch.load(model_path, map_location='cpu')

model.load_state_dict(state_dict, strict=True)
model = model.to(device)
model.eval()

print('checkpoint loaded')


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


checkpoint loaded


In [15]:
# Test Dataset
test_dataset = ReviewPairDataset(
    test['LLaMA_Output'],
    test['LLaVA_Output'],
    test['user_id'].to_numpy(),
    test['asin'].to_numpy(),
    test['rating'].to_numpy(),
    tokenizer,
    max_length=512
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

len(test_dataset), len(test_loader)


(13344, 105)

## Test 데이터 평가

모든 Test 행의 예측값을 저장한 뒤 MAE, MSE, RMSE, MAPE를 계산합니다.


In [16]:
# Test 데이터 전체 예측
prediction_list = []
rating_list = []

with torch.inference_mode():
    for batch in tqdm(test_loader, desc='Test Prediction'):
        labels = batch.pop('labels')
        batch = {key: value.to(device, non_blocking=True) for key, value in batch.items()}

        outputs = model(**batch)

        prediction_list.append(outputs['logits'].detach().cpu().numpy())
        rating_list.append(labels.numpy())

predictions = np.concatenate(prediction_list)
ratings = np.concatenate(rating_list)

MAE = mean_absolute_error(ratings, predictions)
MSE = mean_squared_error(ratings, predictions)
RMSE = np.sqrt(MSE)
MAPE = mean_absolute_percentage_error(ratings, predictions) * 100

print(f'MAE: {MAE:.4f}')
print(f'MSE: {MSE:.4f}')
print(f'RMSE: {RMSE:.4f}')
print(f'MAPE: {MAPE:.4f}%')


Test Prediction:   0%|          | 0/105 [00:00<?, ?it/s]

MAE: 0.3278
MSE: 0.2664
RMSE: 0.5161
MAPE: 9.0362%


In [17]:
# 전체 예측 결과 저장
prediction_df = test[[
    'user_id_raw',
    'asin_raw',
    'rating',
    'title_y',
    'LLaMA_Output',
    'LLaVA_Output'
]].copy()

prediction_df.insert(0, 'source_index', test_source_index)
prediction_df['prediction'] = predictions
prediction_df['absolute_error'] = np.abs(
    prediction_df['rating'] - prediction_df['prediction']
)
prediction_df['signed_error'] = (
    prediction_df['prediction'] - prediction_df['rating']
)

user_count = df.groupby('user_id_raw').size()
item_count = df.groupby('asin_raw').size()

prediction_df['user_interactions'] = prediction_df['user_id_raw'].map(user_count)
prediction_df['item_interactions'] = prediction_df['asin_raw'].map(item_count)

prediction_path = '/content/drive/MyDrive/MAS_Rec/MAS_Rec_test_predictions.pkl'
prediction_df.to_pickle(prediction_path)

print(prediction_df.shape)
print(prediction_path)


(13344, 12)
/content/drive/MyDrive/MAS_Rec/MAS_Rec_test_predictions.pkl


저장된 예측 파일은 다음 오류 분석 노트북에서 사용합니다. 가중치와 전체 예측 파일은 GitHub에 올리지 않습니다.
